### Simple environment
using random jamming, single action instead of dual for simplicity (1 action , identified by freq * power)


In [ ]:
import gymnasium as gym
import numpy as np
import random as random
class AntiJammingEnv(gym.Env):
    def __init__(self, num_freqs=22, num_powers=50, num_of_jammers=7, jammer_type="random"):
        super().__init__()
        self.num_freqs = num_freqs
        self.num_powers = num_powers
        self.jammer_type = jammer_type
        self.num_of_jammers = num_of_jammers

        # Define action space (freq × power combinations)
        self.action_space = gym.spaces.Discrete(num_freqs * num_powers)

        # State = PSD (vector of size num_freqs)
        self.observation_space = gym.spaces.Box(
            low=0.0, high=1.0, shape=(num_freqs,), dtype=np.float32
        )

        self.prev_freq = None
        self.state = None

        #self.rng = random number generator
        self.rng = np.random.default_rng()

        # Internal jammer state & dynamics parameters
        self._jam_idxs = []             # index of the currently jammed frequency bin
        self._sweep_step = 1             # how many bins to move per step for sweeping jammer

    def _build_state_from_jammer(self):
        """
        Construct the PSD observation vector given the *current* jammers indecies.
        - Background noise is simulated as small random values (0..0.1).
        - One bin (self._jam_idx) is set to 1.0 to represent the jammer’s power.
        
        This function is used in both `reset()` and `step()` so that
        the observation is always generated consistently.
        """
        s = self.rng.random(self.num_freqs) * 0.1   # background noise
        if len(self._jam_idxs) == 0:                   # if no jammer index yet, pick a random one
            self._jam_idxs = random.sample(range(self.num_freqs), self.num_of_jammers)
        for i in self._jam_idxs:
            s[i] = 1.0

        return s.astype(np.float32)

    def _advance_jammer(self):
        """
        Update the jammer index based on jammer_type dynamics:
        - fixed: jammer never moves
        - sweeping: jammer shifts to the next bin each step
        - random: jammer picks a new random bin every step
        """
        jt = self.jammer_type.lower()
        if jt == "fixed":
            pass  # jammer stays in the same bin forever
        elif jt == "sweeping":
            for i in range(len(self._jam_idxs)):
                self._jam_idxs[i] = (self._jam_idxs[i] + self._sweep_step) % self.num_freqs
            #self._jam_idx = (self._jam_idx + self._sweep_step) % self.num_freqs
        else:  # default "random"
            for i in range(len(self._jam_idxs)):
                self._jam_idxs[i] = int(self.rng.integers(0, self.num_freqs))
            #self._jam_idx = int(self.rng.integers(0, self.num_freqs))

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        # Initialize jammer index and build the first observation
        self._jam_idxs = random.sample(range(self.num_freqs), self.num_of_jammers)
        self.state = self._build_state_from_jammer()

        self.prev_freq = None
        info = {}
        return self.state.astype(np.float32), info

    def step(self, action):
        freq = action // self.num_powers
        power = action % self.num_powers + 1 # scale 1..num_powers

        # --- Calculate reward based on *current* PSD (self.state) ---
        noise_at_chosen = self.state[freq]
        snr = power / (noise_at_chosen + 1e-6)
        crc_success = 1 if snr > 1.5 else 0     # success if SNR exceeds threshold

        # -- Switching penalty ---
        penalty = 0
        if self.prev_freq is not None and self.prev_freq != freq:
            penalty = 0.2                       # discourage frequent switching
            
        # --- Reward calculation ---
        reward = crc_success * (np.log(1 + snr) / power) - penalty

        # -- Update previous frequency ---
        self.prev_freq = freq

        # Advance jammer dynamics for the *next* state
        self._advance_jammer()
        self.state = self._build_state_from_jammer()

        terminated = False                      # task-specific terminal condition could go here
        truncated = False                       # e.g. max steps per episode
        info = {}
        return self.state.astype(np.float32), reward, terminated, truncated, info
def step(self, action):
        """
        One-slot delay logic:
        - Agent picked `action` using observation s_k (= self._last_psd).
        - Environment first advances jammer to unseen slot k+1 and measures s_{k+1}.
        - Reward is computed against s_{k+1} (the actual slot you just transmitted in).
        - Next observation returned to agent is s_{k+1}.
        """
        # Decode (freq, power)
        freq = action // self.num_powers
        power = action % self.num_powers + 1  # 1..num_powers

        # === Move to *new* (unseen) slot and measure PSD_{k+1} ===
        self._advance_jammer()
        curr_psd = self._build_state_from_jammer()   # s_{k+1}

        # --- Compute reward using the *current* slot conditions (which were not observed before acting) ---
        noise_at_chosen = curr_psd[freq]
        snr = power / (noise_at_chosen + 1e-6)
        crc_success = 1 if snr > 1.5 else 0

        # Switching penalty (compare with last used freq)
        penalty = 0.0
        if self.prev_freq is not None and self.prev_freq != freq:
            penalty = 0.2

        # Reward (kept your original form)
        reward = crc_success * (np.log(1 + snr) / power) - penalty

        # Update previous freq
        self.prev_freq = freq

        # === One-slot-delay book-keeping ===
        # Next observation is the PSD we just measured in this slot.
        self._last_psd = curr_psd
        self.state = curr_psd

        terminated = False
        truncated = False
        info = {}
        return self.state.astype(np.float32), reward, terminated, truncated, info

### instantiate env and agent (DQN) to learn for 200_000 steps

In [ ]:
from stable_baselines3 import DQN

env = AntiJammingEnv()
env = gym.wrappers.TimeLimit(env, max_episode_steps=15)  # <-- K


model = DQN("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=200_000, log_interval=4)


### create a greedy policy (no RL in greedy, just to see if RL works)
decode action is a helper function to return freq and power from single action space of freq * power

In [ ]:
import numpy as np

def decode_action(a, num_powers):
    freq = a // num_powers
    power = (a % num_powers) + 1
    return int(freq), int(power)

def greedy_policy(obs, num_powers, snr_thresh=1.5):
    # pick the least noisy freq (min PSD), then the smallest power that clears the threshold
    f = int(np.argmin(obs))
    for p in range(1, num_powers+1):
        snr = p / (float(obs[f]) + 1e-6)
        if snr > snr_thresh:
            return f * num_powers + (p - 1)
    return f * num_powers + (num_powers - 1)


In [ ]:
def evaluate_agent(env, predict_fn, num_episodes=50, horizon=128, deterministic=True, name="agent"):
    returns = []
    success_rates = []
    switch_rates = []
    avg_powers = []
    jam_hit_rates = []  # how often we transmit on a jammed bin (≈ obs[f] > 0.9 in your env)

    for _ in range(num_episodes):
        obs, _ = env.reset()
        prev_freq = None
        total_r = 0.0
        success = 0
        switches = 0
        power_used = 0
        jam_hits = 0

        for t in range(horizon):
            # action from policy
            if name == "DQN":
                a, _ = predict_fn(obs, deterministic=deterministic) # DQN model chooses next action
            else:  # greedy baseline uses a function of obs
                a = predict_fn(obs)

            # step
            obs_next, r, terminated, truncated, info = env.step(int(a))
            total_r += float(r)

            # decode & compute metrics from obs_next (the env uses this for reward)
            freq, power = decode_action(int(a), env.num_powers)

            # Signal to noise ratio calculation (SNR) at chosen freq
            snr = power / (float(obs_next[freq]) + 1e-6)
            if snr > 1.5:
                success += 1

            if prev_freq is not None and prev_freq != freq:
                switches += 1

            # Penalize switching in reward calculation above
            prev_freq = freq

            power_used += power
            if obs_next[freq] > 0.9:  # jammed bin in this env is set to exactly 1.0
                jam_hits += 1

            if terminated or truncated:
                break

            obs = obs_next

        returns.append(total_r)
        success_rates.append(success / horizon)
        switch_rates.append(switches / horizon)
        avg_powers.append(power_used / horizon)
        jam_hit_rates.append(jam_hits / horizon)

    print(f"{name}: "
          f"return={np.mean(returns):.3f} "
          f"| success={np.mean(success_rates):.3f} "
          f"| switch_rate={np.mean(switch_rates):.3f} "
          f"| avg_power={np.mean(avg_powers):.2f} "
          f"| jam_hit_rate={np.mean(jam_hit_rates):.3f}")


In [ ]:
# DQN (your trained model)
test_env = AntiJammingEnv()
evaluate_agent(test_env, model.predict, num_episodes=50, horizon=128, name="DQN")

# Greedy baseline
evaluate_agent(AntiJammingEnv(), lambda o: greedy_policy(o, num_powers=5), num_episodes=50, horizon=128, name="Greedy")
